# Image batch filter

Generate a batch of small synthetic RGB images on the client, then ship each image to a worker that applies a CPU-heavy Sobel edge filter (numpy only, no Pillow). The workers return the filtered image plus a small summary statistic; the client renders a grid of input-vs-output thumbnails.

Pattern showcased: batch processing where each task carries its own per-task input blob. The filter kernel is real arithmetic, not a trivial lookup, so the wall-clock time scales with worker count.

**Prerequisites**: a Scaler scheduler reachable from this kernel. See :doc:`../tutorials/try_in_browser` for the recommended worker setup.

In [ ]:
# Connection settings -- edit these to point at your running cluster.
SCHEDULER_ADDRESS = "ws://127.0.0.1:2345"  # supports tcp:// or ws://; only ws:// works from JupyterLite (browser)
OBJECT_STORAGE_ADDRESS = None  # leave None to use whatever the scheduler advertises

GRID_SIDE = 4              # display GRID_SIDE x GRID_SIDE input/output thumbnails
IMAGE_SIZE = 128           # each image is IMAGE_SIZE x IMAGE_SIZE x 3 uint8 ~= 48 KiB

In [ ]:
import time

import numpy as np

from scaler import Client


def _synthetic_image(seed: int, size: int) -> np.ndarray:
    """Client-side: small textured RGB image; cheap to build so we never block the browser."""
    rng = np.random.default_rng(seed)
    base = rng.integers(0, 256, size=(size, size, 3), dtype=np.uint8)
    xs = np.linspace(0, 4 * np.pi, size)
    grad = (128 + 127 * np.sin(xs + seed)).astype(np.uint8)
    base[..., 0] = np.clip(base[..., 0] // 2 + grad[np.newaxis, :], 0, 255)
    base[..., 1] = np.clip(base[..., 1] // 2 + grad[:, np.newaxis], 0, 255)
    return base


def sobel_edges(image: np.ndarray) -> tuple[np.ndarray, float]:
    """Worker-side: classic Sobel edge filter on a grayscale projection."""
    luminance = (0.299 * image[..., 0] + 0.587 * image[..., 1] + 0.114 * image[..., 2]).astype(np.float32)
    kx = np.array([[-1.0, 0.0, 1.0], [-2.0, 0.0, 2.0], [-1.0, 0.0, 1.0]], dtype=np.float32)
    ky = kx.T
    h, w = luminance.shape
    gx = np.zeros_like(luminance)
    gy = np.zeros_like(luminance)
    for dy in range(-1, 2):
        for dx in range(-1, 2):
            shifted = np.roll(np.roll(luminance, dy, axis=0), dx, axis=1)
            gx += kx[dy + 1, dx + 1] * shifted
            gy += ky[dy + 1, dx + 1] * shifted
    magnitude = np.hypot(gx, gy)
    magnitude *= 255.0 / max(magnitude.max(), 1.0)
    edges = magnitude.astype(np.uint8)
    return edges, float(edges.mean())


n_images = GRID_SIDE * GRID_SIDE
inputs = [_synthetic_image(seed, IMAGE_SIZE) for seed in range(n_images)]

with Client(address=SCHEDULER_ADDRESS, object_storage_address=OBJECT_STORAGE_ADDRESS) as client:
    started = time.perf_counter()
    futures = [client.submit(sobel_edges, img) for img in inputs]
    results = [f.result() for f in futures]
    elapsed = time.perf_counter() - started

edges = [edge for edge, _stat in results]
mean_intensities = [stat for _edge, stat in results]
print(f"filtered {n_images} {IMAGE_SIZE}x{IMAGE_SIZE} images in {elapsed:.2f}s")
print(f"mean edge intensity per image (first 4): {[round(v, 1) for v in mean_intensities[:4]]}")

In [ ]:
# Display the input/output grid. Client-side only and intentionally cheap.
import matplotlib.pyplot as plt

fig, axes = plt.subplots(GRID_SIDE, 2 * GRID_SIDE, figsize=(2 * GRID_SIDE * 1.3, GRID_SIDE * 1.3))
for idx in range(n_images):
    row = idx // GRID_SIDE
    col = idx % GRID_SIDE
    axes[row, 2 * col].imshow(inputs[idx])
    axes[row, 2 * col].set_axis_off()
    axes[row, 2 * col + 1].imshow(edges[idx], cmap="gray")
    axes[row, 2 * col + 1].set_axis_off()
fig.suptitle("input (left) vs. Sobel edges (right)")
fig.tight_layout()
plt.show()